# 9 WorkFlow Analista Jr

### 9.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán
<br>El Analista Jr corre sus scripts en la virtual manchine **desktop-jr** que tiene estas características


*   Normal, paga tarifa completa, nunca es apagada por Google
*   reside en el datacenter de Toronto, Canada
*   64 GB de memoria RAM
*   8 vCPU


En Analista Jr **no** puede utilizar Google Colab porque los 12 GB de dichas maquinas virtuales no son suficientes para el tamaño del dataset que está utilizando.



## 9.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Wed Sep 09 10:51:26 2026"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671155,35.9,1479540,79.1,1479540,79.1
Vcells,1242565,9.5,8388608,64.0,1978697,15.1


In [3]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, u

#### Parametros

In [4]:
PARAM <- list()
PARAM$semilla_primigenia <- 100313

PARAM$experimento <- 9200
PARAM$dataset <- "analistajr_competencia_2026.csv.gz"

#### Carpeta del Experimento

In [5]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

### 9.3.1   Preprocesamiento del dataset

#### 9.3.1.1  DT incorporar dataset

In [6]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

#### 9.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [7]:
if( !require("mice")) install.packages("mice", repos = "http://cran.us.r-project.org")
require("mice")

Loading required package: mice


Attaching package: ‘mice’


The following object is masked from ‘package:stats’:

    filter


The following objects are masked from ‘package:base’:

    cbind, rbind




In [8]:
# Escrito por alumnos de  Universidad Austral  Rosario

Corregir_MICE <- function(pcampo, pmeses) {

  meth <- rep("", ncol(dataset))
  names(meth) <- colnames(dataset)
  meth[names(meth) == pcampo] <- "sample"

  # llamada a mice  !
  imputacion <- mice(dataset,
    method = meth,
    maxit = 5,
    m = 1,
    seed = 7)

  tbl <- mice::complete(dataset)

  dataset[, paste0(pcampo) := ifelse(foto_mes %in% pmeses, tbl[, get(pcampo)], get(pcampo))]

}


In [9]:
Corregir_interpolar <- function(pcampo, pmeses) {

  tbl <- dataset[, list(
    "v1" = shift(get(pcampo), 1, type = "lag"),
    "v2" = shift(get(pcampo), 1, type = "lead")
  ),
  by = eval(envg$PARAM$dataset_metadata$entity_id)
  ]

  tbl[, paste0(envg$PARAM$dataset_metadata$entity_id) := NULL]
  tbl[, promedio := rowMeans(tbl, na.rm = TRUE)]

  dataset[
    ,
    paste0(pcampo) := ifelse(!(foto_mes %in% pmeses),
      get(pcampo),
      tbl$promedio
    )
  ]
}

In [10]:
AsignarNA_campomeses <- function(pcampo, pmeses) {

  if( pcampo %in% colnames( dataset ) ) {

    dataset[ foto_mes %in% pmeses, paste0(pcampo) := NA ]
  }
}

In [11]:

Corregir_atributo <- function(pcampo, pmeses, pmetodo)
{
  # si el campo no existe en el dataset, Afuera !
  if( !(pcampo %in% colnames( dataset )) )
    return( 1 )

  # llamo a la funcion especializada que corresponde
  switch( pmetodo,
    "MachineLearning"     = AsignarNA_campomeses(pcampo, pmeses),
    "EstadisticaClasica"  = Corregir_interpolar(pcampo, pmeses),
    "MICE"                = Corregir_MICE(pcampo, pmeses),
  )

  return( 0 )
}

In [12]:

Corregir_Rotas <- function(dataset, pmetodo) {
  gc(verbose= FALSE)
  cat( "inicio Corregir_Rotas()\n")
  # acomodo los errores del dataset

  Corregir_atributo("active_quarter", c(202006), pmetodo) # 1
  Corregir_atributo("internet", c(202006), pmetodo) # 2

  Corregir_atributo("mrentabilidad", c(201905, 201910, 202006), pmetodo) # 3
  Corregir_atributo("mrentabilidad_annual", c(201905, 201910, 202006), pmetodo) # 4

  Corregir_atributo("mcomisiones", c(201905, 201910, 202006), pmetodo) # 5

  Corregir_atributo("mactivos_margen", c(201905, 201910, 202006), pmetodo) # 6
  Corregir_atributo("mpasivos_margen", c(201905, 201910, 202006), pmetodo) # 7

  Corregir_atributo("mcuentas_saldo", c(202006), pmetodo) # 8

  Corregir_atributo("ctarjeta_debito_transacciones", c(202006), pmetodo) # 9

  Corregir_atributo("mautoservicio", c(202006), pmetodo) # 10

  Corregir_atributo("ctarjeta_visa_transacciones", c(202006), pmetodo) # 11
  Corregir_atributo("mtarjeta_visa_consumo", c(202006), pmetodo) # 12

  Corregir_atributo("ctarjeta_master_transacciones", c(202006), pmetodo) # 13
  Corregir_atributo("mtarjeta_master_consumo", c(202006), pmetodo) # 14

  Corregir_atributo("ctarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 15
  Corregir_atributo("mttarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 16

  Corregir_atributo("ccajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 17

  Corregir_atributo("mcajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 18

  Corregir_atributo("ctarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 19

  Corregir_atributo("mtarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 20

  Corregir_atributo("ctarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 21

  Corregir_atributo("mtarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 22

  Corregir_atributo("ccomisiones_otras", c(201905, 201910, 202006), pmetodo) # 23
  Corregir_atributo("mcomisiones_otras", c(201905, 201910, 202006), pmetodo) # 24

  Corregir_atributo("cextraccion_autoservicio", c(202006), pmetodo) # 25
  Corregir_atributo("mextraccion_autoservicio", c(202006), pmetodo) # 26

  Corregir_atributo("ccheques_depositados", c(202006), pmetodo) # 27
  Corregir_atributo("mcheques_depositados", c(202006), pmetodo) # 28
  Corregir_atributo("ccheques_emitidos", c(202006), pmetodo) # 29
  Corregir_atributo("mcheques_emitidos", c(202006), pmetodo) # 30
  Corregir_atributo("ccheques_depositados_rechazados", c(202006), pmetodo) # 31
  Corregir_atributo("mcheques_depositados_rechazados", c(202006), pmetodo) # 32
  Corregir_atributo("ccheques_emitidos_rechazados", c(202006), pmetodo) # 33
  Corregir_atributo("mcheques_emitidos_rechazados", c(202006), pmetodo) # 34

  Corregir_atributo("tcallcenter", c(202006), pmetodo) # 35
  Corregir_atributo("ccallcenter_transacciones", c(202006), pmetodo) # 36

  Corregir_atributo("thomebanking", c(202006), pmetodo) # 37
  Corregir_atributo("chomebanking_transacciones", c(201910, 202006), pmetodo) # 38

  Corregir_atributo("ccajas_transacciones", c(202006), pmetodo) # 39
  Corregir_atributo("ccajas_consultas", c(202006), pmetodo) # 40

  Corregir_atributo("ccajas_depositos", c(202006, 202105), pmetodo) # 41

  Corregir_atributo("ccajas_extracciones", c(202006), pmetodo) # 41
  Corregir_atributo("ccajas_otras", c(202006), pmetodo) # 43

  Corregir_atributo("catm_trx", c(202006), pmetodo) # 44
  Corregir_atributo("matm", c(202006), pmetodo) # 45
  Corregir_atributo("catm_trx_other", c(202006), pmetodo) # 46
  Corregir_atributo("matm_other", c(202006), pmetodo) # 47

  cat( "fin Corregir_rotas()\n")
}


In [13]:
# resuelvo el Catastrophe Analysis

setorder( dataset, numero_de_cliente, foto_mes )

PARAM$CA$metodo= "MachineLearning"

if( PARAM$CA$metodo %in% c("MachineLearning", "EstadisticaClasica", "MICE") )
  Corregir_Rotas(dataset, PARAM$CA$metodo)

inicio Corregir_Rotas()
fin Corregir_rotas()


#### 9.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, ajustando por algunos indices financieros

In [14]:
# meses que me interesan para el ajuste de variables monetarias
vfoto_mes <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107, 202108, 202109
)


In [15]:
# los valores que siguen fueron calculados por alumnos

# momento 1.0  31-dic-2020 a las 23:59
vIPC <- c(
  1.9903030878, 1.9174403544, 1.8296186587,
  1.7728862972, 1.7212488323, 1.6776304408,
  1.6431248196, 1.5814483345, 1.4947526791,
  1.4484037589, 1.3913580777, 1.3404220402,
  1.3154288912, 1.2921698342, 1.2472681797,
  1.2300475145, 1.2118694724, 1.1881073259,
  1.1693969743, 1.1375456949, 1.1065619600,
  1.0681100000, 1.0370000000, 1.0000000000,
  0.9680542110, 0.9344152616, 0.8882274350,
  0.8532444140, 0.8251880213, 0.8003763543,
  0.7763107219, 0.7566381305, 0.7289384687
)

vdolar_blue <- c(
   39.045455,  38.402500,  41.639474,
   44.274737,  46.095455,  45.063333,
   43.983333,  54.842857,  61.059524,
   65.545455,  66.750000,  72.368421,
   77.477273,  78.191667,  82.434211,
  101.087500, 126.236842, 125.857143,
  130.782609, 133.400000, 137.954545,
  170.619048, 160.400000, 153.052632,
  157.900000, 149.380952, 143.615385,
  146.250000, 153.550000, 162.000000,
  178.478261, 180.878788, 184.357143
)

vdolar_oficial <- c(
   38.430000,  39.428000,  42.542105,
   44.354211,  46.088636,  44.955000,
   43.751429,  54.650476,  58.790000,
   61.403182,  63.012632,  63.011579,
   62.983636,  63.580556,  65.200000,
   67.872000,  70.047895,  72.520952,
   75.324286,  77.488500,  79.430909,
   83.134762,  85.484737,  88.181667,
   91.474000,  93.997778,  96.635909,
   98.526000,  99.613158, 100.619048,
  101.619048, 102.569048, 103.781818
)

vUVA <- c(
  2.001408838932958,  1.950325472789153,  1.89323032351521,
  1.8247220405493787, 1.746027787673673,  1.6871348409529485,
  1.6361678865622313, 1.5927529755859773, 1.5549162794128493,
  1.4949100586391746, 1.4197729500774545, 1.3678188186372326,
  1.3136508617223726, 1.2690535173062818, 1.2381595983200178,
  1.211656735577568,  1.1770808941405335, 1.1570338657445522,
  1.1388769475653255, 1.1156993751209352, 1.093638313080772,
  1.0657171590878205, 1.0362173587708712, 1.0,
  0.9669867858358365, 0.9323750098728378, 0.8958202912590305,
  0.8631993702994263, 0.8253893405524657, 0.7928918905364516,
  0.7666323845128089, 0.7428976357662823, 0.721615762047849
)


In [16]:
tb_indices <- as.data.table( list(
  "IPC" = vIPC,
  "dolar_blue" = vdolar_blue,
  "dolar_oficial" = vdolar_oficial,
  "UVA" = vUVA
  )
)

tb_indices[[ 'foto_mes' ]] <- vfoto_mes

tb_indices

IPC,dolar_blue,dolar_oficial,UVA,foto_mes
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1.9903031,39.04545,38.43000,2.0014088,201901
1.9174404,38.40250,39.42800,1.9503255,201902
1.8296187,41.63947,42.54210,1.8932303,201903
1.7728863,44.27474,44.35421,1.8247220,201904
1.7212488,46.09546,46.08864,1.7460278,201905
1.6776304,45.06333,44.95500,1.6871348,201906
1.6431248,43.98333,43.75143,1.6361679,201907
1.5814483,54.84286,54.65048,1.5927530,201908
1.4947527,61.05952,58.79000,1.5549163,201909


In [17]:
drift_UVA <- function(campos_monetarios) {
  cat( "inicio drift_UVA()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.UVA,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_UVA()\n")
}


In [18]:
drift_dolar_oficial <- function(campos_monetarios) {
  cat( "inicio drift_dolar_oficial()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_oficial,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_oficial()\n")
}


In [19]:
drift_dolar_blue <- function(campos_monetarios) {
  cat( "inicio drift_dolar_blue()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_blue,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_blue()\n")
}


In [20]:
drift_deflacion <- function(campos_monetarios) {
  cat( "inicio drift_deflacion()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.IPC,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_deflacion()\n")
}


In [21]:
drift_rank_simple <- function(campos_drift) {

  cat( "inicio drift_rank_simple()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_rank") :=
      (frank(get(campo), ties.method = "random") - 1) / (.N - 1), by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat( "fin drift_rank_simple()\n")
}


In [22]:
# El cero se transforma en cero
# los positivos se rankean por su lado
# los negativos se rankean por su lado

drift_rank_cero_fijo <- function(campos_drift) {

  cat( "inicio drift_rank_cero_fijo()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[get(campo) == 0, paste0(campo, "_rank") := 0]
    dataset[get(campo) > 0, paste0(campo, "_rank") :=
      frank(get(campo), ties.method = "random") / .N, by = list(foto_mes)]

    dataset[get(campo) < 0, paste0(campo, "_rank") :=
      -frank(-get(campo), ties.method = "random") / .N, by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat("\n")
  cat( "fin drift_rank_cero_fijo()\n")
}


In [23]:
drift_estandarizar <- function(campos_drift) {

  cat( "inicio drift_estandarizar()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_normal") :=
      (get(campo) -mean(campo, na.rm=TRUE)) / sd(get(campo), na.rm=TRUE),
      by = list(foto_mes)]

    dataset[, (campo) := NULL]
  }
  cat( "fin drift_estandarizar()\n")
}


In [24]:
# por como armé los nombres de campos,
#  estos son los campos que expresan variables monetarias
campos_monetarios <- colnames(dataset)
campos_monetarios <- campos_monetarios[campos_monetarios %like%
  "^(m|Visa_m|Master_m|vm_m)"]

campos_monetarios

[1] "mrentabilidad"                      "mrentabilidad_annual"              
 [3] "mcomisiones"                        "mactivos_margen"                   
 [5] "mpasivos_margen"                    "mcuenta_corriente"                 
 [7] "mcaja_ahorro"                       "mcuentas_saldo"                    
 [9] "mtarjeta_visa_consumo"              "mtarjeta_master_consumo"           
[11] "mprestamos_personales"              "mpayroll"                          
[13] "mttarjeta_visa_debitos_automaticos" "mcomisiones_mantenimiento"         
[15] "mtransferencias_recibidas"          "Master_mfinanciacion_limite"       
[17] "Master_msaldototal"                 "Master_mlimitecompra"              
[19] "Master_mconsumototal"               "Master_mpagominimo"                
[21] "Visa_mfinanciacion_limite"          "Visa_msaldototal"                  
[23] "Visa_mlimitecompra"                 "Visa_mconsumototal"                
[25] "Visa_mpagominimo"

In [25]:
# ejecuto el Data Drifting
setorder( dataset, numero_de_cliente, foto_mes )


PARAM$DR$metodo <- "deflacion"

switch(PARAM$DR$metodo,
  "ninguno"        = cat("No hay correccion del data drifting"),
  "rank_simple"    = drift_rank_simple(campos_monetarios),
  "rank_cero_fijo" = drift_rank_cero_fijo(campos_monetarios),
  "deflacion"      = drift_deflacion(campos_monetarios),
  "dolar_blue"     = drift_dolarblue(campos_monetarios),
  "dolar_oficial"  = drift_dolaroficial(campos_monetarios),
  "UVA"            = drift_UVA(campos_monetarios),
  "estandarizar"   = drift_estandarizar(campos_monetarios)
)


inicio drift_deflacion()
fin drift_deflacion()


In [26]:
colnames(dataset)

[1] "numero_de_cliente"                  "foto_mes"                          
 [3] "internet"                           "cliente_edad"                      
 [5] "cliente_antiguedad"                 "mrentabilidad"                     
 [7] "mrentabilidad_annual"               "mcomisiones"                       
 [9] "mactivos_margen"                    "mpasivos_margen"                   
[11] "cproductos"                         "mcuenta_corriente"                 
[13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
[15] "mcuentas_saldo"                     "ctarjeta_visa"                     
[17] "ctarjeta_visa_transacciones"        "mtarjeta_visa_consumo"             
[19] "ctarjeta_master"                    "ctarjeta_master_transacciones"     
[21] "mtarjeta_master_consumo"            "cprestamos_personales"             
[23] "mprestamos_personales"              "cpayroll_trx"                      
[25] "mpayroll"                           "mttarjeta_visa_debitos_automaticos"
[27] "ccomisiones_mantenimiento"          "mcomisiones_mantenimiento"         
[29] "ccomisiones_otras"                  "mtransferencias_recibidas"         
[31] "ccallcenter_transacciones"          "thomebanking"                      
[33] "chomebanking_transacciones"         "ctrx_quarter"                      
[35] "Master_status"                      "Master_mfinanciacion_limite"       
[37] "Master_Fvencimiento"                "Master_msaldototal"                
[39] "Master_mlimitecompra"               "Master_fultimo_cierre"             
[41] "Master_fechaalta"                   "Master_mconsumototal"              
[43] "Master_cconsumos"                   "Master_mpagominimo"                
[45] "Visa_status"                        "Visa_mfinanciacion_limite"         
[47] "Visa_Fvencimiento"                  "Visa_msaldototal"                  
[49] "Visa_mlimitecompra"                 "Visa_fultimo_cierre"               
[51] "Visa_fechaalta"                     "Visa_mconsumototal"                
[53] "Visa_cconsumos"                     "Visa_mpagominimo"                  
[55] "clase_ternaria"

In [27]:
# se intenta corregir el data drifting utilizando algunos indices financieros

#### 9.3.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [28]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# variable extraida de una tesis de maestria de Irlanda
if( atributos_presentes( c("mpayroll", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]


##### Inicio GA

#### =================================================================
#### PROBLEMA #10: Feature Engineering Intra-mes - Algoritmo Genetico
#### Modalidad Analista Jr / Sr - Grupo A
####
#### HIPOTESIS EXPERIMENTAL:
####   "Un algoritmo genetico que vaya generando las combinaciones de
####    variables y descarte aquellas que no sean de utilidad debe ser
####    mucho mejor que las variables que se nos puedan ocurrir o que
####    la bibliografia acerca del sector financiero pueda plantear."
####
#### ARQUITECTURA (2 etapas, cada una con su paquete):
####  - Etapa 1 - gramEvol: GENERA un pool de formulas candidatas
####             (evolucion gramatical, fitness barato = correlacion)
####  - Etapa 2 - GA:       SELECCIONA el subconjunto optimo del pool
####             (algoritmo genetico binario, fitness = AUC real)
#### IMPORTANTE: en este punto del pipeline TODAVIA NO EXISTEN
####   fold_train, PARAM$training, PARAM$validate (se crean recien en
####   9.3.2.1 Training Strategy). Por eso ambas etapas usan un split
####   LOCAL propio, cuidando siempre de no tocar el mes 202107 (que
####   mas adelante sera el validate oficial del pipeline).
#### =================================================================

In [29]:
# 0. Definición de parametros
library(gramEvol)
library(lightgbm)
library(data.table)
library(parallel)

if (!"clase01" %in% colnames(dataset)) {
  dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1, 0)]
}

excluir_GA <- c("numero_de_cliente", "foto_mes", "clase_ternaria", "clase01", "azar")
candidatas_GA <- setdiff(colnames(dataset), excluir_GA)

# me quedo solo con columnas numericas
son_numericas <- sapply(dataset[, candidatas_GA, with = FALSE], is.numeric)
candidatas_GA <- candidatas_GA[son_numericas]

# excluyo variables de fecha (ver nota arriba)
patron_fechas <- "^(f|.*_f|.*fecha)"
candidatas_GA <- candidatas_GA[!grepl(patron_fechas, candidatas_GA, ignore.case = TRUE)]

# limite de seguridad para no explotar el espacio de busqueda
MAX_TERMINALES <- 40
if (length(candidatas_GA) > MAX_TERMINALES) {
  set.seed(PARAM$semilla_primigenia)
  candidatas_GA <- sample(candidatas_GA, MAX_TERMINALES)
}

cat("Variables candidatas para el algoritmo genetico:", length(candidatas_GA), "\n")
print(candidatas_GA)

cols_candidatas <- candidatas_GA

Variables candidatas para el algoritmo genetico: 40 
 [1] "ccallcenter_transacciones"          "kmes"                              
 [3] "ctarjeta_master"                    "mcuentas_saldo"                    
 [5] "cdescubierto_preacordado"           "internet"                          
 [7] "Visa_cconsumos"                     "mcaja_ahorro"                      
 [9] "Visa_mconsumototal"                 "mcuenta_corriente"                 
[11] "mprestamos_personales"              "mtarjeta_master_consumo"           
[13] "cliente_antiguedad"                 "Visa_mfinanciacion_limite"         
[15] "Visa_mpagominimo"                   "cprestamos_personales"             
[17] "mtarjeta_visa_consumo"              "Visa_status"                       
[19] "mcomisiones_mantenimiento"          "cliente_edad"                      
[21] "Visa_mlimitecompra"                 "ccomisiones_mantenimiento"         
[23] "Master_mpagominimo"                 "Master_cconsumos"                  

In [30]:
# =================================================================
# Problema #10 - Version SANEADA del script alternativo
# (basado en BNF texto + fitness = LightGBM univariado real)
#
# BUGS ENCONTRADOS Y CORREGIDOS EN ESTA VERSION:
#  1. fitness_gramEvol llamaba a GrammarMap(expr, ...) de mas.
#     'expr' YA llega mapeado por GrammaticalEvolution() -> se saco.
#  2. El chequeo any(!is.finite(valores)) descartaba con UN SOLO NA,
#     borrando de un plumazo TODAS las variables relacionadas con
#     Visa (que tienen NA estructural en clientes sin esa tarjeta).
#     -> ahora solo se descarta si TODOS los valores son NA/Inf,
#     o si no hay variabilidad entre los validos.
#  3. El Paso 4 (extraccion del Top 5) reutilizaba fitness_gramEvol
#     pasandole el GENOMA CRUDO (poblacion_final[[i]]) en vez de la
#     expresion mapeada. eval() sobre un vector de enteros no tira
#     error, devuelve el vector tal cual -> entrenaba LightGBM con
#     el genoma en bruto como si fuera una variable real (numeros
#     sin sentido, pero un AUC "valido" que no mide nada real).
#     -> ahora se mapea con GrammarMap() ANTES de evaluar.
# =================================================================

library(gramEvol)
library(lightgbm)
library(data.table)
library(parallel)
TOP_N <- 20

# ================================================================
# 0. Setup: split local (excluye 202107 en adelante, igual que en
#    el resto del trabajo, para no tocar el validate oficial ni
#    los meses sin clase_ternaria completa: 202108 solo BAJA+1,
#    202109 vacio)
# ================================================================
if (!"clase01" %in% colnames(dataset)) {
  dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1, 0)]
}

meses_disponibles <- sort(unique(dataset$foto_mes[dataset$foto_mes < 202107]))
meses_val <- tail(meses_disponibles, 3)
meses_tr  <- setdiff(meses_disponibles, meses_val)

idx_train <- which(dataset$foto_mes %in% meses_tr)
idx_valid <- which(dataset$foto_mes %in% meses_val)
y_train <- dataset$clase01[idx_train]
y_valid <- dataset$clase01[idx_valid]

cat("Meses de entrenamiento:", length(meses_tr), " | Meses de validacion:", length(meses_val), "\n")
cat("(meses de validacion usados:", paste(meses_val, collapse=", "), ")\n")

# ================================================================
# 1. Gramatica BNF (formato texto)
# ================================================================
protected_div <- function(x, y) {
  res <- x / y
  res[is.na(res) | is.infinite(res)] <- 0
  res
}
protected_log_diff <- function(x, y) {
  res <- log(abs(x - y) + 1)
  res[is.na(res) | is.infinite(res)] <- 0
  res
}

string_vars <- paste(candidatas_GA, collapse = " | ")
rule_text <- paste0(
  "<expr> ::= <op>\n",
  "<op>   ::= <op> + <op> | <op> - <op> | <op> * <op> | ",
  "protected_div(<op>, <op>) | protected_log_diff(<op>, <op>) | <var>\n",
  "<var>  ::= ", string_vars
)
tf <- tempfile()
writeLines(rule_text, tf)
bnf_grammar <- CreateGrammar(tf)
unlink(tf)

# ================================================================
# 2. Fitness: LightGBM univariado real (BUG 1 y 2 corregidos)
#    Recibe la expresion YA MAPEADA (no el genoma).
# ================================================================
fitness_gramEvol <- function(expr) {
  valores <- tryCatch(eval(expr, envir = dataset), error = function(e) NULL)
  if (is.null(valores)) return(1)  # costo máximo si falla

  valores_finitos <- valores[is.finite(valores)]
  if (length(valores_finitos) == 0 || length(unique(valores_finitos)) <= 1) {
    return(1)  # costo máximo, no 0
  }

  dtr  <- lgb.Dataset(data = matrix(valores[idx_train], ncol = 1), label = y_train)
  dval <- lgb.Dataset(data = matrix(valores[idx_valid], ncol = 1), label = y_valid)

  modelo <- tryCatch({
    lgb.train(
      params = list(objective = "binary", metric = "auc",
                    learning_rate = 0.1, num_threads = 1, verbosity = -1),
      data = dtr, valids = list(valid = dval),
      nrounds = 50, early_stopping_rounds = 10, verbose = -1
    )
  }, error = function(e) NULL)

  if (is.null(modelo) || is.null(modelo$best_score) || is.na(modelo$best_score)) return(1)

  1 - modelo$best_score   # <-- COSTO: menor es mejor, AUC alto -> costo bajo
}

# Funcion auxiliar NUEVA: mapea un GENOMA crudo -> expresion -> fitness.
# Se usa en el Paso 4, donde SI corresponde llamar a GrammarMap()
# (a diferencia de dentro de fitness_gramEvol, donde no correspondia).
evaluar_genoma <- function(genoma) {
  expr_obj <- tryCatch(suppressWarnings(GrammarMap(genoma, bnf_grammar)), error = function(e) NULL)
  if (is.null(expr_obj) || !isTRUE(GrammarIsTerminal(expr_obj))) {
    return(list(score = 0, formula = NA_character_))
  }
  expr_lang <- tryCatch(as.expression(expr_obj), error = function(e) NULL)
  if (is.null(expr_lang) || length(expr_lang) == 0) {
    return(list(score = 0, formula = NA_character_))
  }

  expr_final  <- expr_lang[[1]]
  formula_str <- paste(deparse(expr_final), collapse = "")
  costo <- fitness_gramEvol(expr_final)
  auc   <- 1 - costo   # reconvierto costo -> AUC para reportar/ordenar

  list(score = auc, formula = formula_str)
}


# ================================================================
# 4. Ejecucion del Algoritmo Genetico
# ================================================================
set.seed(PARAM$semilla_primigenia)

cat("
=== Iniciando GrammaticalEvolution ===
")
ge_res <- GrammaticalEvolution(
  grammarDef      = bnf_grammar,
  evalFunc        = fitness_gramEvol,
  popSize         = 200,     # VOLVER a 200 (o más). Con 20 es imposible sacar 20 features
  iterations      = 50,      # más generaciones para explorar
  terminationCost = 0,       # NO cortar por umbral → forzar exploración completa
  seqLen          = 200,
  max.depth       = 10,
  mutationChance  = 0.2,
  elitism         = 0.05,
  monitorFunc     = function(result) {
    cat(sprintf("Gen %d | Best cost: %.5f (AUC: %.5f)",
                result$population$currentIteration,
                result$best$cost,
                1 - result$best$cost))
  }
)

# ================================================================
# 5. Extraccion del Top 5
#    FIX: se combinan DOS fuentes de candidatos:
#      (a) el/los mejores GLOBALES que gramEvol garantiza en ge_res$best
#      (b) la poblacion de la ultima generacion (como antes)
#    Asi no dependemos solo de la ultima generacion, que puede NO
#    contener el mejor individuo historico.
# ================================================================
cat("
=== Evaluando candidatos para extraer el Top 5 ===
")

# --- (a) Mejores GLOBALES que reporta gramEvol -------------------
# ge_res$best puede traer una o varias expresiones ya mapeadas.
# Las convierto a formula_str + AUC con el MISMO fitness real.
mejores_globales <- list()

agregar_expr_global <- function(expr_obj) {
  # expr_obj ya es una expresion mapeada (no genoma)
  expr_lang <- tryCatch(as.expression(expr_obj), error = function(e) NULL)
  if (is.null(expr_lang) || length(expr_lang) == 0) return(NULL)
  expr_final  <- expr_lang[[1]]
  formula_str <- paste(deparse(expr_final), collapse = "")
  costo <- fitness_gramEvol(expr_final)   # AUC real, no genoma crudo
  list(score = 1 - costo, formula = formula_str)
}

# ge_res$best$expressions es lo mas comun; contemplo variantes
if (!is.null(ge_res$best)) {
  cand_best <- ge_res$best$expressions
  if (is.null(cand_best) && !is.null(ge_res$best$expression)) {
    cand_best <- list(ge_res$best$expression)
  }
  if (!is.null(cand_best)) {
    if (!is.list(cand_best)) cand_best <- list(cand_best)
    for (e in cand_best) {
      r <- tryCatch(agregar_expr_global(e), error = function(err) NULL)
      if (!is.null(r)) mejores_globales[[length(mejores_globales) + 1]] <- r
    }
  }
}
cat(sprintf("Mejores globales recuperados de ge_res$best: %d",
            length(mejores_globales)))

# --- (b) Poblacion de la ultima generacion (como antes) ----------
pop_matrix      <- ge_res$population$population
poblacion_final <- split(pop_matrix, row(pop_matrix))

n_cores <- max(1, detectCores() - 1)
cat(sprintf("Evaluando %d individuos de la poblacion final en %d cores...",
            length(poblacion_final), n_cores))

resultados_pop <- mclapply(poblacion_final, evaluar_genoma, mc.cores = n_cores)

# --- Combino ambas fuentes en una sola lista de candidatos -------
resultados <- c(mejores_globales, resultados_pop)

scores_finales   <- sapply(resultados, function(r) r$score)
formulas_finales <- sapply(resultados, function(r) r$formula)

ordenados <- order(scores_finales, decreasing = TRUE)

# ================================================================
# Seleccion del Top N (deduplicando por formula)
# ================================================================
formulas_vistas <- character()
ga_cols_creadas <- character()
top_guardados   <- 0
idx             <- 1

while (top_guardados < TOP_N && idx <= length(ordenados)) {
  i <- ordenados[idx]
  idx <- idx + 1

  if (is.na(scores_finales[i]) || scores_finales[i] <= 0) next
  if (is.na(formulas_finales[i]) || formulas_finales[i] %in% formulas_vistas) next

  eval_res <- tryCatch(
    eval(parse(text = formulas_finales[i])[[1]], envir = dataset),
    error = function(e) NULL
  )
  if (is.null(eval_res) || length(unique(eval_res[is.finite(eval_res)])) <= 1) next

  top_guardados <- top_guardados + 1
  formulas_vistas <- c(formulas_vistas, formulas_finales[i])

  nombre_col <- paste0("GA_Feature_", top_guardados)
  ga_cols_creadas <- c(ga_cols_creadas, nombre_col)

  cat(sprintf("[%s] AUC: %.5f | Formula: %s
",
              nombre_col, scores_finales[i], formulas_finales[i]))
  dataset[, (nombre_col) := eval_res]
}

cat("
Columnas creadas:", paste(ga_cols_creadas, collapse = ", "), "
")

Meses de entrenamiento: 27  | Meses de validacion: 3 
(meses de validacion usados: 202104, 202105, 202106 )

=== Iniciando GrammaticalEvolution ===
Gen 1 | Best cost: 0.21383 (AUC: 0.78617)
Gen 2 | Best cost: 0.21051 (AUC: 0.78949)
Gen 3 | Best cost: 0.17391 (AUC: 0.82609)
Gen 4 | Best cost: 0.17391 (AUC: 0.82609)
Gen 5 | Best cost: 0.17391 (AUC: 0.82609)
Gen 6 | Best cost: 0.17391 (AUC: 0.82609)
Gen 7 | Best cost: 0.17391 (AUC: 0.82609)
Gen 8 | Best cost: 0.17391 (AUC: 0.82609)
Gen 9 | Best cost: 0.17186 (AUC: 0.82814)
Gen 10 | Best cost: 0.17186 (AUC: 0.82814)
Gen 11 | Best cost: 0.17186 (AUC: 0.82814)
Gen 12 | Best cost: 0.17186 (AUC: 0.82814)
Gen 13 | Best cost: 0.17186 (AUC: 0.82814)
Gen 14 | Best cost: 0.17186 (AUC: 0.82814)
Gen 15 | Best cost: 0.17186 (AUC: 0.82814)
Gen 16 | Best cost: 0.17186 (AUC: 0.82814)
Gen 17 | Best cost: 0.17186 (AUC: 0.82814)
Gen 18 | Best cost: 0.17186 (AUC: 0.82814)
Gen 19 | Best cost: 0.17186 (AUC: 0.82814)
Gen 20 | Best cost: 0.17186 (AUC: 0.82814)
G

##### Fin GA

In [32]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)

[1] "numero_de_cliente"                  "foto_mes"                          
 [3] "internet"                           "cliente_edad"                      
 [5] "cliente_antiguedad"                 "mrentabilidad"                     
 [7] "mrentabilidad_annual"               "mcomisiones"                       
 [9] "mactivos_margen"                    "mpasivos_margen"                   
[11] "cproductos"                         "mcuenta_corriente"                 
[13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
[15] "mcuentas_saldo"                     "ctarjeta_visa"                     
[17] "ctarjeta_visa_transacciones"        "mtarjeta_visa_consumo"             
[19] "ctarjeta_master"                    "ctarjeta_master_transacciones"     
[21] "mtarjeta_master_consumo"            "cprestamos_personales"             
[23] "mprestamos_personales"              "cpayroll_trx"                      
[25] "mpayroll"                           "mttarjeta_visa_debitos_automaticos"
[27] "ccomisiones_mantenimiento"          "mcomisiones_mantenimiento"         
[29] "ccomisiones_otras"                  "mtransferencias_recibidas"         
[31] "ccallcenter_transacciones"          "thomebanking"                      
[33] "chomebanking_transacciones"         "ctrx_quarter"                      
[35] "Master_status"                      "Master_mfinanciacion_limite"       
[37] "Master_Fvencimiento"                "Master_msaldototal"                
[39] "Master_mlimitecompra"               "Master_fultimo_cierre"             
[41] "Master_fechaalta"                   "Master_mconsumototal"              
[43] "Master_cconsumos"                   "Master_mpagominimo"                
[45] "Visa_status"                        "Visa_mfinanciacion_limite"         
[47] "Visa_Fvencimiento"                  "Visa_msaldototal"                  
[49] "Visa_mlimitecompra"                 "Visa_fultimo_cierre"               
[51] "Visa_fechaalta"                     "Visa_mconsumototal"                
[53] "Visa_cconsumos"                     "Visa_mpagominimo"                  
[55] "clase_ternaria"                     "kmes"                              
[57] "mpayroll_sobre_edad"                "clase01"                           
[59] "GA_Feature_1"                       "GA_Feature_2"                      
[61] "GA_Feature_3"                       "GA_Feature_4"                      
[63] "GA_Feature_5"

#### 9.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

In [33]:
# No se implementa Feature Engineering a partir de Random Forest

#### 9.3.1.5  FEhist Feature Engineering historico

El Fature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [34]:
# Feature Engineering Historico

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}


Verificacion de los campos recien creados

In [35]:
ncol(dataset)
colnames(dataset)

[1] 303

[1] "numero_de_cliente"                        
  [2] "foto_mes"                                 
  [3] "internet"                                 
  [4] "cliente_edad"                             
  [5] "cliente_antiguedad"                       
  [6] "mrentabilidad"                            
  [7] "mrentabilidad_annual"                     
  [8] "mcomisiones"                              
  [9] "mactivos_margen"                          
 [10] "mpasivos_margen"                          
 [11] "cproductos"                               
 [12] "mcuenta_corriente"                        
 [13] "mcaja_ahorro"                             
 [14] "cdescubierto_preacordado"                 
 [15] "mcuentas_saldo"                           
 [16] "ctarjeta_visa"                            
 [17] "ctarjeta_visa_transacciones"              
 [18] "mtarjeta_visa_consumo"                    
 [19] "ctarjeta_master"                          
 [20] "ctarjeta_master_transacciones"            
 [21] "mtarjeta_master_consumo"                  
 [22] "cprestamos_personales"                    
 [23] "mprestamos_personales"                    
 [24] "cpayroll_trx"                             
 [25] "mpayroll"                                 
 [26] "mttarjeta_visa_debitos_automaticos"       
 [27] "ccomisiones_mantenimiento"                
 [28] "mcomisiones_mantenimiento"                
 [29] "ccomisiones_otras"                        
 [30] "mtransferencias_recibidas"                
 [31] "ccallcenter_transacciones"                
 [32] "thomebanking"                             
 [33] "chomebanking_transacciones"               
 [34] "ctrx_quarter"                             
 [35] "Master_status"                            
 [36] "Master_mfinanciacion_limite"              
 [37] "Master_Fvencimiento"                      
 [38] "Master_msaldototal"                       
 [39] "Master_mlimitecompra"                     
 [40] "Master_fultimo_cierre"                    
 [41] "Master_fechaalta"                         
 [42] "Master_mconsumototal"                     
 [43] "Master_cconsumos"                         
 [44] "Master_mpagominimo"                       
 [45] "Visa_status"                              
 [46] "Visa_mfinanciacion_limite"                
 [47] "Visa_Fvencimiento"                        
 [48] "Visa_msaldototal"                         
 [49] "Visa_mlimitecompra"                       
 [50] "Visa_fultimo_cierre"                      
 [51] "Visa_fechaalta"                           
 [52] "Visa_mconsumototal"                       
 [53] "Visa_cconsumos"                           
 [54] "Visa_mpagominimo"                         
 [55] "clase_ternaria"                           
 [56] "kmes"                                     
 [57] "mpayroll_sobre_edad"                      
 [58] "clase01"                                  
 [59] "GA_Feature_1"                             
 [60] "GA_Feature_2"                             
 [61] "GA_Feature_3"                             
 [62] "GA_Feature_4"                             
 [63] "GA_Feature_5"                             
 [64] "internet_lag1"                            
 [65] "cliente_edad_lag1"                        
 [66] "cliente_antiguedad_lag1"                  
 [67] "mrentabilidad_lag1"                       
 [68] "mrentabilidad_annual_lag1"                
 [69] "mcomisiones_lag1"                         
 [70] "mactivos_margen_lag1"                     
 [71] "mpasivos_margen_lag1"                     
 [72] "cproductos_lag1"                          
 [73] "mcuenta_corriente_lag1"                   
 [74] "mcaja_ahorro_lag1"                        
 [75] "cdescubierto_preacordado_lag1"            
 [76] "mcuentas_saldo_lag1"                      
 [77] "ctarjeta_visa_lag1"                       
 [78] "ctarjeta_visa_transacciones_lag1"         
 [79] "mtarjeta_visa_consumo_lag1"               
 [80] "ctarjeta_master_lag1"                     
 [

#### 9.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  ni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [36]:
# No se implementa la reduccion de la dimensionalidad con canaritos

### 9.3.2 Modelado

#### 9.3.2.1 Training Strategy

Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202109  obviamente completo

* final_train =  [ 201901, 202107 ]  SIN undersampling

* training
   * testing = NO HAY
   * validation =  202107   completo, sin undersampling
   * training = [ 201901, 202105 ]  donde se consideran el 100% de los CONTINUA

In [37]:
PARAM$trainingstrategy$validate <- c(202107)

PARAM$trainingstrategy$training <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105
)


PARAM$trainingstrategy$training_pct <- 1.0


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [38]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [39]:
# los campos en los que se entrena
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar"))
)

In [40]:
# preparo para que se puede hacer undersampling de los CONTINUA
#  solamente por un tema de VELOCIDAD
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar:=runif(nrow(dataset))]

# undersampling de los CONTINUA
dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct ) ]


if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label= dataset[fold_train == TRUE, clase01],
  free_raw_data= TRUE
)

In [41]:
# datos de validation
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= TRUE
)

nrow(dvalidate)

[1] 32938

####  9.3.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Parametros no default, fijos de LightGBM que no se optimizan
  * max_bin = 31 , Alienigenas Ancestrales contruyeron las pirámides y dejaron a la humanidad en un jeroglifico  *max_bin=31*
  * feature_fraction = 0.5  para poner algo que generalmente no falla
  * learning_rate = 0.03  para que aprenda lento


* Parametros que se optimizan en el Grid Search
  * num_leaves  [64, 512]
  * min_data_in_leaf  [64, 2048]

In [42]:
# parametros fijos del LightGBM
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  seed= PARAM$semilla_primigenia,
  max_bin= 31,
  learning_rate= 0.03,
  feature_fraction= 0.5,
  num_iterations= 2048,  # valor grande, lo limita early_stopping_rounds
  early_stopping_rounds= 200,
  num_leaves= 64,
  min_data_in_leaf= 128
)


In [43]:
# En  x llegan los parametros moviles de LightGBM
#  devuelve la AUC en validate del modelo entrenado
#  en el parametro x llegan los hiperparámetros que se estan optimizando

Estimar_AUC_lightgbm <- function(x) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM
  modelo_train <- lgb.train(
    data= dtrain,
    valids= list(valid = dvalidate),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  # recupero la AUC en validation
  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " niter ", modelo_train$best_iter,
    " AUC ", AUC
  )

  # hago espacio en la memoria
  niter <- modelo_train$best_iter
  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return( list(AUC, niter))
}

seteo del Grid Search

In [44]:
# lo que sigue a continuacion es una forma alternativa a los loops anidados
# creo una tabla con el producto cartesiano de los vectores
tb_nueva <- CJ(
  num_leaves= c(64, 128, 256, 384, 512),
  min_data_in_leaf= c(64, 256, 512, 1024, 2048),
  feature_fraction= c(0.5, 0.8)
)

##### Corrida del Grid Search,  aqui se hace el trabajo pesado
<br> por favor no se asuste con los warnings que pudieran aparecer
<br> ATENCION, la siguiente celda demora 65 minutos
<br> una Analista Jr  debe ser capaz de tolerar estoicamente esta tortura
<br> (y masticar chicle al mismo tiempo)

In [45]:
# registro a registro calculo la AUC
tb_nueva[,  c("AUC", "num_iterations"):= Estimar_AUC_lightgbm( .SD ),
  by=1:nrow(tb_nueva) ]

Wed Sep 09 11:03:13 2026  64, 64, 0.5 niter 252 AUC 0.999998347866819

Wed Sep 09 11:03:42 2026  64, 64, 0.8 niter 228 AUC 0.999998083525509

Wed Sep 09 11:04:45 2026  64, 256, 0.5 niter 280 AUC 0.9999978191842

Wed Sep 09 11:05:17 2026  64, 256, 0.8 niter 336 AUC 0.999997951354855

Wed Sep 09 11:06:34 2026  64, 512, 0.5 niter 354 AUC 0.999997951354855

Wed Sep 09 11:07:06 2026  64, 512, 0.8 niter 328 AUC 0.999998281781491

Wed Sep 09 11:08:33 2026  64, 1024, 0.5 niter 388 AUC 0.999998149610837

Wed Sep 09 11:09:14 2026  64, 1024, 0.8 niter 378 AUC 0.999998017440182

Wed Sep 09 11:10:02 2026  64, 2048, 0.5 niter 91 AUC 0.999997224416255

Wed Sep 09 11:10:33 2026  64, 2048, 0.8 niter 136 AUC 0.999997620928219

Wed Sep 09 11:11:29 2026  128, 64, 0.5 niter 214 AUC 0.999997951354855

Wed Sep 09 11:11:57 2026  128, 64, 0.8 niter 139 AUC 0.999996563562982

Wed Sep 09 11:12:50 2026  128, 256, 0.5 niter 169 AUC 0.999997951354855

Wed Sep 09 11:13:21 2026  128, 256, 0.8 niter 266 AUC 0.99999775

la optimizacion de hiperparámetros de tipo  Grid Search ha corrido, extraigo los mejores hiperparametros

In [ ]:
tb_nueva

fwrite( tb_nueva,
  file= "tb_grid_search_01.txt",
  sep="\t",
  append= TRUE
)

In [ ]:
setorder( tb_nueva, -AUC)  # ordeno DESCENDENTE por AUC
PARAM$out$lgbm$AUC <- tb_nueva[1, AUC] # en la posicion 1 estan los mejores
PARAM$out$lgbm$mejores_hiperparametros <- as.list( tb_nueva[1] )
PARAM$out$lgbm$mejores_hiperparametros$AUC <- NULL
PARAM$out$lgbm$mejores_hiperparametros

In [ ]:
tb_nueva

### 9.3.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización

In [ ]:
PARAM$trainingstrategy$final_train <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107
)


dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

# creo el dfinal_train en formato  LightGBM
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  free_raw_data= TRUE
)

nrow( dfinal_train) # verifico el tamaño

##### Final Training Hyperparameters

In [ ]:
# uno los parametros fijos y los mejores encontrados de los variables
fijos <- copy(PARAM$lgbm$param_fijos)

# quito lo que optimice en la Bayesian Optimization
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL

# agrego a los hiperparametros fijos los que encontre con la Bayesian Optimization
param_final <- c(fijos, PARAM$out$lgbm$mejores_hiperparametros)

##### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [ ]:
final_model <- lgb.train(
  data= dfinal_train,
  param= param_final,
  verbose= -100
)

In [ ]:
# grabo a disco el modelo en un formato para seres humanos ... ponele ...

lgb.save(final_model, "modelo.txt")

In [ ]:
# ahora imprimo la importancia de variables

tb_importancia <- as.data.table(lgb.importance(final_model))
archivo_importancia <- "impo.txt"

fwrite( tb_importancia,
  file= archivo_importancia,
  sep= "\t"
)

#### Scoring

Aplico el modelo final a los datos del futuro

In [ ]:
PARAM$trainingstrategy$future <- c(202109)

dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]

In [ ]:
# aplico final_model   a dfuture

prediccion <- predict(
  final_model,
  data.matrix(dfuture[, campos_buenos, with= FALSE])
)

##### Tabla Prediccion

In [ ]:
tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := prediccion]

# grabo las probabilidad del modelo
#  me va a ser util para hacer Ensembles de modelos
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)

#### Kaggle Competition Submit

Genero las salidas y hago los submits a Kaggle

In [ ]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle

PARAM$kaggle$competencia <- "utn-2026-virtual-jr"
PARAM$kaggle$cortes <- seq(1800, 2400, by = 100)

# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle", showWarnings= FALSE)

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marclo los primeros

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  # subida a Kaggle, armo la linea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios,
  "  semilla=", PARAM$semilla_primigenia,
    "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  salida <- system(linea, intern=TRUE) # el submit a Kaggle
  Sys.sleep(30)
  cat(salida, "\n")
}

In [ ]:
# grabo los parametros
if( !require("yaml")) install.packages("yaml")
require("yaml")

write_yaml( PARAM, file="PARAM.yml")

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")